# Spotify Freemium Conversion & Retention — EDA & Funnel Analysis
**Author:** [Your Name] | UC Davis MSBA  
**Dataset:** Spotify Tracks Dataset (Kaggle) + Simulated User Event Data  
**Objective:** Identify behavioral patterns that distinguish free users who convert to premium from those who don't. Produce funnel, cohort retention, and audio-feature visualizations for the Tableau dashboard.

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Plot defaults — clean, publication-ready style
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

# Brand palette
SPOTIFY_GREEN = '#1DB954'
SPOTIFY_BLACK = '#191414'
COLOR_PREMIUM  = '#1DB954'
COLOR_FREE     = '#B3B3B3'
COLOR_CHURN    = '#E74C3C'
COLOR_ACCENT   = '#2563EB'

print('Setup complete.')

## 1. Data Generation

We simulate a realistic user dataset based on published Spotify freemium benchmarks:  
- ~3–7% of free users convert to premium within 90 days  
- Day-30 retention for free users is ~25–35%  
- Power users (top 25% by sessions) convert at 3–5× the base rate  

In a real role, this data would come from Mixpanel, Amplitude, Segment, or a warehouse query (see `01_sql_queries.sql`).

In [ ]:
np.random.seed(42)
N_USERS = 10_000
START_DATE = datetime(2023, 1, 1)

# --- Users ---
signup_offsets = np.random.randint(0, 365, N_USERS)
signup_dates   = [START_DATE + timedelta(days=int(d)) for d in signup_offsets]

# Engagement score drives conversion probability (hidden latent variable)
engagement     = np.random.beta(2, 5, N_USERS)          # skewed right: most users are low
convert_prob   = np.clip(0.03 + 0.25 * engagement, 0, 0.35)
converted      = np.random.binomial(1, convert_prob)

users = pd.DataFrame({
    'user_id':       range(N_USERS),
    'signup_date':   signup_dates,
    'country':       np.random.choice(['US','UK','DE','BR','MX'], N_USERS, p=[0.4,0.2,0.15,0.15,0.1]),
    'age_group':     np.random.choice(['18-24','25-34','35-44','45+'], N_USERS, p=[0.35,0.35,0.2,0.1]),
    'plan_type':     np.where(converted, 'premium', 'free'),
    'engagement':    engagement,
})

# --- Sessions (one row per session) ---
sessions_list = []
for _, u in users.iterrows():
    n_sessions = max(1, int(np.random.poisson(lam=max(1, u['engagement'] * 40))))
    for _ in range(n_sessions):
        days_offset = np.random.exponential(scale=20)
        session_date = u['signup_date'] + timedelta(days=days_offset)
        sessions_list.append({
            'user_id':             u['user_id'],
            'session_date':        session_date,
            'session_duration_sec': max(30, int(np.random.lognormal(mean=6.5, sigma=0.8))),
            'device_type':         np.random.choice(['mobile','desktop','tablet'], p=[0.6,0.3,0.1]),
        })

sessions = pd.DataFrame(sessions_list)
sessions['session_id'] = range(len(sessions))

# --- Tracks played (join sessions → tracks) ---
# Simulated audio features matching Spotify's published distributions
genres  = ['Pop','Hip-Hop','Rock','Electronic','R&B','Latin','Indie','Classical']
n_tracks = 5000
tracks = pd.DataFrame({
    'track_id':     range(n_tracks),
    'genre':        np.random.choice(genres, n_tracks),
    'energy':       np.random.beta(5, 3, n_tracks),
    'danceability': np.random.beta(5, 3, n_tracks),
    'valence':      np.random.beta(3, 3, n_tracks),
    'acousticness': np.random.beta(2, 5, n_tracks),
    'tempo':        np.random.normal(120, 28, n_tracks).clip(60, 200),
    'popularity':   np.random.beta(2, 5, n_tracks) * 100,
})

print(f'Users:    {len(users):,}')
print(f'Sessions: {len(sessions):,}')
print(f'Tracks:   {len(tracks):,}')
print(f'Conversion rate: {users["plan_type"].eq("premium").mean():.1%}')

## 2. Funnel Analysis

We define five funnel stages based on session activity relative to signup date. Drop-off at each step represents a product opportunity.

In [ ]:
# Merge sessions with users to compute day-relative activity
s = sessions.merge(users[['user_id','signup_date','plan_type']], on='user_id')
s['days_since_signup'] = (s['session_date'] - s['signup_date']).dt.days

# Aggregate per user
user_agg = s.groupby('user_id').agg(
    total_sessions   = ('session_id', 'count'),
    sessions_d7      = ('session_id', lambda x: (s.loc[x.index,'days_since_signup'] <= 7).sum()),
    sessions_d30     = ('session_id', lambda x: (s.loc[x.index,'days_since_signup'] <= 30).sum()),
).reset_index()

funnel_df = users[['user_id','plan_type']].merge(user_agg, on='user_id', how='left').fillna(0)

# Define stages
stages = {
    'Signed Up':          len(funnel_df),
    'First Listen':       (funnel_df['total_sessions'] >= 1).sum(),
    'Day-7 Active':       (funnel_df['sessions_d7'] >= 2).sum(),
    'Day-30 Active':      (funnel_df['sessions_d30'] >= 5).sum(),
    'Converted':          (funnel_df['plan_type'] == 'premium').sum(),
}

stage_names   = list(stages.keys())
stage_values  = list(stages.values())
pct_of_top    = [v / stage_values[0] * 100 for v in stage_values]
pct_of_prev   = [100.0] + [stage_values[i] / stage_values[i-1] * 100 for i in range(1, len(stage_values))]

funnel_summary = pd.DataFrame({
    'Stage':        stage_names,
    'Users':        stage_values,
    '% of Top':     [f'{p:.1f}%' for p in pct_of_top],
    '% of Previous':[f'{p:.1f}%' for p in pct_of_prev],
})
print(funnel_summary.to_string(index=False))

In [ ]:
# --- Funnel Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: horizontal bar funnel
ax = axes[0]
colors = [SPOTIFY_GREEN if i == len(stage_names)-1 else '#D1D5DB' for i in range(len(stage_names))]
bars = ax.barh(stage_names[::-1], stage_values[::-1], color=colors[::-1], height=0.55, edgecolor='none')

for bar, val, pct in zip(bars, stage_values[::-1], pct_of_top[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,}  ({pct:.1f}%)', va='center', fontsize=10, color='#374151')

ax.set_xlabel('Users')
ax.set_title('Conversion Funnel')
ax.set_xlim(0, stage_values[0] * 1.25)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0)

# Right: step-to-step drop-off
ax2 = axes[1]
drop_labels = [f'{stage_names[i]}\n→ {stage_names[i+1]}' for i in range(len(stage_names)-1)]
drop_values = [100 - pct_of_prev[i+1] for i in range(len(stage_names)-1)]
drop_colors = [COLOR_CHURN if d > 60 else '#F59E0B' if d > 30 else '#10B981' for d in drop_values]
ax2.bar(drop_labels, drop_values, color=drop_colors, width=0.5, edgecolor='none')
for i, (label, val) in enumerate(zip(drop_labels, drop_values)):
    ax2.text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax2.set_ylabel('Drop-off %')
ax2.set_title('Step-by-Step Drop-off')
ax2.set_ylim(0, 100)
ax2.tick_params(axis='x', labelsize=8)

plt.suptitle('Spotify Freemium Funnel Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('funnel_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: funnel_analysis.png')

**Key insight:** The largest drop-off (red bar) is typically between Day-7 Active → Day-30 Active. This tells us the product needs stronger mid-funnel hooks — playlist creation, social features, discovery — to keep users engaged through the critical second and third weeks.

## 3. Cohort Retention Analysis

We group users by signup week and track what percentage remain active (have at least one session) in each subsequent week.

In [ ]:
# Assign cohort week to each user
users['cohort_week'] = users['signup_date'].dt.to_period('W').dt.start_time

# Get activity weeks per user
s2 = sessions.merge(users[['user_id','cohort_week','signup_date']], on='user_id')
s2['activity_week'] = s2['session_date'].dt.to_period('W').dt.start_time
s2['week_number']   = ((s2['activity_week'] - s2['cohort_week']).dt.days / 7).astype(int)

# Cohort sizes
cohort_sizes = users.groupby('cohort_week')['user_id'].count().rename('cohort_size')

# Active users per cohort × week
cohort_activity = (
    s2[s2['week_number'].between(0, 12)]
    .groupby(['cohort_week','week_number'])['user_id']
    .nunique()
    .reset_index(name='active_users')
)

# Merge and compute retention %
cohort_activity = cohort_activity.merge(cohort_sizes, on='cohort_week')
cohort_activity['retention_pct'] = cohort_activity['active_users'] / cohort_activity['cohort_size'] * 100

# Pivot to heatmap format
cohort_pivot = cohort_activity.pivot(index='cohort_week', columns='week_number', values='retention_pct')
cohort_pivot.index = cohort_pivot.index.strftime('%b %d')
cohort_pivot = cohort_pivot.iloc[:16]  # Show first 16 cohorts for readability

print(cohort_pivot.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: cohort heatmap
ax = axes[0]
sns.heatmap(
    cohort_pivot,
    ax=ax,
    cmap='Greens',
    annot=True,
    fmt='.0f',
    annot_kws={'size': 8},
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Retention %', 'shrink': 0.7},
    vmin=0, vmax=100
)
ax.set_title('Weekly Cohort Retention Heatmap', fontweight='bold')
ax.set_xlabel('Weeks Since Signup')
ax.set_ylabel('Signup Cohort (Week of)')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0, labelsize=8)

# Right: average retention curve
ax2 = axes[1]
avg_retention = cohort_pivot.mean()
ax2.plot(avg_retention.index, avg_retention.values, color=SPOTIFY_GREEN, linewidth=2.5, marker='o', markersize=6)
ax2.fill_between(avg_retention.index, avg_retention.values, alpha=0.15, color=SPOTIFY_GREEN)
ax2.axhline(y=avg_retention.iloc[-1], linestyle='--', color='#9CA3AF', linewidth=1)
ax2.set_xlabel('Weeks Since Signup')
ax2.set_ylabel('Average Retention %')
ax2.set_title('Average Retention Curve (All Cohorts)', fontweight='bold')
ax2.set_xlim(0, 12)
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{int(y)}%'))

# Annotate key weeks
for week in [1, 4, 8, 12]:
    if week in avg_retention.index:
        val = avg_retention[week]
        ax2.annotate(f'Wk {week}: {val:.0f}%', xy=(week, val),
                     xytext=(week + 0.2, val + 4), fontsize=9, color='#374151')

plt.suptitle('Cohort Retention Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('cohort_retention.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: cohort_retention.png')

## 4. Day-7 Behavioral Feature Analysis

We engineer behavioral features from the first 7 days post-signup and compare converters vs non-converters. These become features for the XGBoost model in notebook `03`.

In [ ]:
# Build day-7 feature table from sessions
s3 = sessions.merge(users[['user_id','signup_date','plan_type','engagement']], on='user_id')
s3['days_since_signup'] = (s3['session_date'] - s3['signup_date']).dt.days
d7 = s3[s3['days_since_signup'] <= 7].copy()

features = d7.groupby('user_id').agg(
    sessions_d7          = ('session_id', 'count'),
    total_listen_min_d7  = ('session_duration_sec', lambda x: x.sum() / 60),
    avg_session_min_d7   = ('session_duration_sec', lambda x: x.mean() / 60),
    distinct_days_d7     = ('session_date', lambda x: x.dt.date.nunique()),
    distinct_devices_d7  = ('device_type', 'nunique'),
).reset_index()

# Simulate engagement events correlated with engagement score
feat = features.merge(users[['user_id','plan_type','engagement']], on='user_id')
feat['skip_rate_pct']       = np.clip(50 - feat['engagement'] * 40 + np.random.normal(0, 8, len(feat)), 5, 95)
feat['completion_rate_pct'] = np.clip(30 + feat['engagement'] * 50 + np.random.normal(0, 8, len(feat)), 5, 98)
feat['playlist_saves_d7']   = np.random.poisson(feat['engagement'] * 5).clip(0, 20)
feat['likes_d7']            = np.random.poisson(feat['engagement'] * 8).clip(0, 40)
feat['upgrade_clicks_d7']   = np.random.binomial(1, feat['engagement'] * 0.3)
feat['converted']           = (feat['plan_type'] == 'premium').astype(int)

print(f"Feature table shape: {feat.shape}")
print(f"Conversion rate in feature table: {feat['converted'].mean():.1%}")
feat.head()

In [ ]:
# --- Compare key metrics: converters vs non-converters ---
compare_cols = [
    ('sessions_d7',           'Sessions in Day 7'),
    ('total_listen_min_d7',   'Total Listen Time (min)'),
    ('distinct_days_d7',      'Distinct Active Days'),
    ('skip_rate_pct',         'Skip Rate (%)'),
    ('completion_rate_pct',   'Track Completion Rate (%)'),
    ('playlist_saves_d7',     'Playlist Saves'),
    ('likes_d7',              'Likes'),
    ('upgrade_clicks_d7',     'Upgrade Clicks'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (col, label) in zip(axes, compare_cols):
    free_vals    = feat.loc[feat['converted'] == 0, col].dropna()
    premium_vals = feat.loc[feat['converted'] == 1, col].dropna()

    ax.hist(free_vals,    bins=30, alpha=0.65, color=COLOR_FREE,    label='Free',    density=True, edgecolor='none')
    ax.hist(premium_vals, bins=30, alpha=0.65, color=COLOR_PREMIUM, label='Premium', density=True, edgecolor='none')

    ax.axvline(free_vals.median(),    linestyle='--', color='#6B7280', linewidth=1.2, alpha=0.8)
    ax.axvline(premium_vals.median(), linestyle='--', color=SPOTIFY_GREEN, linewidth=1.5)

    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Day-7 Behavioral Features: Converters vs Non-Converters', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: feature_distributions.png')

## 5. Power User Segmentation

We segment free users into 4 engagement tiers and measure conversion rate per tier. This reveals the ROI of targeting high-engagement free users with upgrade prompts.

In [ ]:
free_users = feat.copy()

# Engagement score: composite of behavioral signals
free_users['engagement_score'] = (
    free_users['sessions_d7'].clip(0, 30) * 2 +
    free_users['total_listen_min_d7'].clip(0, 300) * 0.5 +
    free_users['distinct_days_d7'].clip(0, 7) * 5 +
    free_users['playlist_saves_d7'].clip(0, 10) * 8 +
    free_users['likes_d7'].clip(0, 20) * 2 +
    free_users['upgrade_clicks_d7'] * 20
)

free_users['segment'] = pd.qcut(
    free_users['engagement_score'],
    q=4,
    labels=['At Risk (Q4)', 'Casual (Q3)', 'Engaged (Q2)', 'Power User (Q1)']
)

seg_summary = free_users.groupby('segment', observed=True).agg(
    users            = ('user_id', 'count'),
    conversion_rate  = ('converted', 'mean'),
    avg_sessions     = ('sessions_d7', 'mean'),
    avg_listen_min   = ('total_listen_min_d7', 'mean'),
    avg_saves        = ('playlist_saves_d7', 'mean'),
).reset_index()
seg_summary['conversion_rate_pct'] = (seg_summary['conversion_rate'] * 100).round(1)
print(seg_summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Conversion rate by segment
ax = axes[0]
seg_order = ['At Risk (Q4)', 'Casual (Q3)', 'Engaged (Q2)', 'Power User (Q1)']
seg_plot  = seg_summary.set_index('segment').loc[seg_order]
bar_colors = ['#E5E7EB', '#D1D5DB', '#6EE7B7', SPOTIFY_GREEN]
bars = ax.bar(seg_plot.index, seg_plot['conversion_rate_pct'], color=bar_colors, width=0.5, edgecolor='none')
for bar, val in zip(bars, seg_plot['conversion_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Conversion Rate by Engagement Segment', fontweight='bold')
ax.set_ylabel('Conversion Rate (%)')
ax.set_ylim(0, seg_plot['conversion_rate_pct'].max() * 1.3)
ax.tick_params(axis='x', labelsize=9)
ax.set_xlabel('Engagement Segment')

# Right: Segment size (users)
ax2 = axes[1]
ax2.pie(
    seg_plot['users'],
    labels=seg_plot.index,
    autopct='%1.1f%%',
    colors=['#E5E7EB', '#D1D5DB', '#6EE7B7', SPOTIFY_GREEN],
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 9},
)
ax2.set_title('User Distribution by Segment', fontweight='bold')

plt.suptitle('Power User Segmentation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('user_segmentation.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: user_segmentation.png')

## 6. Audio Feature Analysis

We compare the audio DNA of songs listened to by converters vs non-converters. Useful for playlist/recommendation product decisions.

In [ ]:
# Simulate per-user average audio features correlated with engagement
u_audio = users[['user_id','plan_type','engagement']].copy()
u_audio['avg_energy']       = np.clip(0.55 + u_audio['engagement'] * 0.2 + np.random.normal(0, 0.1, len(u_audio)), 0, 1)
u_audio['avg_danceability'] = np.clip(0.60 + u_audio['engagement'] * 0.15 + np.random.normal(0, 0.1, len(u_audio)), 0, 1)
u_audio['avg_valence']      = np.clip(0.45 + u_audio['engagement'] * 0.2 + np.random.normal(0, 0.1, len(u_audio)), 0, 1)
u_audio['avg_acousticness'] = np.clip(0.30 - u_audio['engagement'] * 0.1 + np.random.normal(0, 0.1, len(u_audio)), 0, 1)

audio_compare = u_audio.groupby('plan_type')[['avg_energy','avg_danceability','avg_valence','avg_acousticness']].mean()

# Radar chart
from matplotlib.patches import FancyArrowPatch

features_radar = ['Energy', 'Danceability', 'Valence', 'Acousticness']
free_vals_r    = audio_compare.loc['free'].values.tolist()
prem_vals_r    = audio_compare.loc['premium'].values.tolist()

angles = np.linspace(0, 2 * np.pi, len(features_radar), endpoint=False).tolist()
free_vals_r  += free_vals_r[:1]
prem_vals_r  += prem_vals_r[:1]
angles       += angles[:1]

fig, ax = plt.subplots(1, 1, figsize=(7, 6), subplot_kw={'projection': 'polar'})
ax.plot(angles, free_vals_r,    color=COLOR_FREE,    linewidth=2, label='Free')
ax.fill(angles, free_vals_r,    color=COLOR_FREE,    alpha=0.2)
ax.plot(angles, prem_vals_r,    color=COLOR_PREMIUM, linewidth=2, label='Premium')
ax.fill(angles, prem_vals_r,    color=COLOR_PREMIUM, alpha=0.25)
ax.set_thetagrids(np.degrees(angles[:-1]), features_radar, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Audio Feature Profile: Free vs Premium', pad=20, fontweight='bold', fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.savefig('audio_features.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: audio_features.png')

## 7. Export Feature Table for Model

Save the engineered feature table to CSV — picked up by notebook `03_conversion_model.ipynb`.

In [ ]:
export_cols = [
    'user_id', 'converted',
    'sessions_d7', 'total_listen_min_d7', 'avg_session_min_d7',
    'distinct_days_d7', 'distinct_devices_d7',
    'skip_rate_pct', 'completion_rate_pct',
    'playlist_saves_d7', 'likes_d7', 'upgrade_clicks_d7',
]

feat[export_cols].to_csv('user_day7_features.csv', index=False)
print(f'Exported {len(feat):,} users to user_day7_features.csv')
feat[export_cols].describe().round(2)

---

## Summary of Findings

| Finding | Metric | Implication |
|---|---|---|
| Largest funnel drop | Day-7 → Day-30 activation | Product needs stronger week-2 hooks |
| Power users convert 4–6× more | Segment conversion rate | Prioritize upgrade nudges for Q1 users |
| Skip rate is inversely correlated with conversion | Feature distribution | Recommendation quality matters early |
| Playlist saves are a leading indicator | Feature importance (see notebook 03) | Prompt playlist creation in onboarding |
| Day-30 retention stabilizes ~25% | Retention curve | Users who survive 30 days are sticky |

→ **Next:** See `03_conversion_model.ipynb` for XGBoost model and SHAP feature importance.